<a href="https://colab.research.google.com/github/Val-Faria/TechChallengeFase4/blob/main/TechChallengeFase4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%pip install -q azure-cognitiveservices-speech

In [8]:
from google.colab import userdata
import azure.cognitiveservices.speech as speechsdk

SPEECH_KEY = userdata.get("SPEECH_KEY")
SPEECH_REGION = userdata.get("SPEECH_REGION") or "eastus"

speech_config = speechsdk.SpeechConfig(
    subscription=SPEECH_KEY,
    region=SPEECH_REGION
)

speech_config.speech_synthesis_voice_name = "pt-BR-FranciscaNeural"

In [9]:
import os
import random
import html
import time
import pandas as pd

random.seed(42)

PASTA_ORIGINAL = "/content/dataset_audio_emocoes"
os.makedirs(PASTA_ORIGINAL, exist_ok=True)

classes = {
    "ansiedade_hesitacao": {
        "rate": "-10%",
        "pitch": "+10%",
        "volume": "default"
    },
    "neutro": {
        "rate": "0%",
        "pitch": "0%",
        "volume": "default"
    },
    "tristeza": {
        "rate": "-20%",
        "pitch": "-10%",
        "volume": "soft"
    }
}

frases = [
    "Estou me sentindo bem hoje",
    "Estou um pouco preocupado com o procedimento",
    "Não tenho certeza se isso está normal",
    "Estou sentindo um desconforto",
    "Acho que consigo continuar",
    "Estou com medo de que algo aconteça",
    "Minha respiração está normal",
    "Estou me sentindo cansado",
    "Preciso de ajuda para ficar mais tranquilo",
    "Estou com dificuldade para relaxar",
    "Sinto uma pressão no peito",
    "Estou tentando manter a calma",
    "Não sei explicar exatamente o que estou sentindo",
    "Estou muito nervoso neste momento",
    "A dor está diminuindo",
    "Estou me sentindo triste",
    "Estou sem disposição hoje",
    "Não estou me sentindo muito bem",
    "Acho que preciso descansar",
    "Está tudo bem, podemos continuar"
]

prefixos = {
    "ansiedade_hesitacao": [
        "",
        "Bem, ",
        "Hum, ",
        "Eu acho que ",
        "Talvez "
    ],
    "neutro": [
        "",
        "Certo, ",
        "Sim, ",
        "Agora, ",
        "Neste momento, "
    ],
    "tristeza": [
        "",
        "Infelizmente, ",
        "Bom, ",
        "Neste momento, ",
        "Eu sinto que "
    ]
}

In [10]:
registros = []

for classe, estilo in classes.items():

    combinacoes = [
        prefixo + frase
        for frase in frases
        for prefixo in prefixos[classe]
    ]

    random.shuffle(combinacoes)
    combinacoes = combinacoes[:100]

    for indice, texto in enumerate(combinacoes):

        texto_ssml = html.escape(texto)

        if classe == "ansiedade_hesitacao":
            texto_ssml = texto_ssml.replace(
                "acho que",
                "acho que <break time='400ms'/>"
            ).replace(
                "Talvez",
                "Talvez <break time='500ms'/>"
            )

        elif classe == "tristeza":
            texto_ssml = texto_ssml.replace(
                " ",
                " <break time='100ms'/> ",
                2
            )

        ssml = f"""
        <speak version="1.0"
               xmlns="http://www.w3.org/2001/10/synthesis"
               xml:lang="pt-BR">
            <voice name="pt-BR-FranciscaNeural">
                <prosody rate="{estilo['rate']}"
                         pitch="{estilo['pitch']}"
                         volume="{estilo['volume']}">
                    {texto_ssml}
                </prosody>
            </voice>
        </speak>
        """

        nome = f"{classe}_{indice:03d}.wav"
        caminho = os.path.join(PASTA_ORIGINAL, nome)

        audio_config = speechsdk.audio.AudioOutputConfig(
            filename=caminho
        )

        sintetizador = speechsdk.SpeechSynthesizer(
            speech_config=speech_config,
            audio_config=audio_config
        )

        resultado = sintetizador.speak_ssml_async(ssml).get()

        if resultado.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
            tamanho = os.path.getsize(caminho)

            if tamanho > 0:
                registros.append({
                    "arquivo": nome,
                    "caminho": caminho,
                    "classe": classe,
                    "texto": texto,
                    "origem": "azure_speech_sintetico"
                })

                print("OK:", nome)

        else:
            detalhes = resultado.cancellation_details
            print("ERRO:", nome)
            print("Motivo:", detalhes.reason)
            print("Código:", detalhes.error_code)
            print("Detalhes:", detalhes.error_details)

        time.sleep(0.3)

df_original = pd.DataFrame(registros)

df_original.to_csv(
    os.path.join(PASTA_ORIGINAL, "metadata_original.csv"),
    index=False,
    encoding="utf-8"
)

print(df_original["classe"].value_counts())

OK: ansiedade_hesitacao_000.wav
OK: ansiedade_hesitacao_001.wav
OK: ansiedade_hesitacao_002.wav
OK: ansiedade_hesitacao_003.wav
OK: ansiedade_hesitacao_004.wav
OK: ansiedade_hesitacao_005.wav
OK: ansiedade_hesitacao_006.wav
OK: ansiedade_hesitacao_007.wav
OK: ansiedade_hesitacao_008.wav
OK: ansiedade_hesitacao_009.wav
OK: ansiedade_hesitacao_010.wav
OK: ansiedade_hesitacao_011.wav
OK: ansiedade_hesitacao_012.wav
OK: ansiedade_hesitacao_013.wav
OK: ansiedade_hesitacao_014.wav
OK: ansiedade_hesitacao_015.wav
OK: ansiedade_hesitacao_016.wav
OK: ansiedade_hesitacao_017.wav
OK: ansiedade_hesitacao_018.wav
OK: ansiedade_hesitacao_019.wav
OK: ansiedade_hesitacao_020.wav
OK: ansiedade_hesitacao_021.wav
OK: ansiedade_hesitacao_022.wav
OK: ansiedade_hesitacao_023.wav
OK: ansiedade_hesitacao_024.wav
OK: ansiedade_hesitacao_025.wav
OK: ansiedade_hesitacao_026.wav
OK: ansiedade_hesitacao_027.wav
OK: ansiedade_hesitacao_028.wav
OK: ansiedade_hesitacao_029.wav
OK: ansiedade_hesitacao_030.wav
OK: ansi

In [11]:
!pip -q install librosa soundfile

In [12]:
import librosa
import soundfile as sf
import numpy as np
import shutil

PASTA_FINAL = "/content/dataset_audio_emocoes_1200"
os.makedirs(PASTA_FINAL, exist_ok=True)

registros_finais = []

for _, linha in df_original.iterrows():

    audio, sr = librosa.load(
        linha["caminho"],
        sr=16000,
        mono=True
    )

    base = os.path.splitext(linha["arquivo"])[0]

    variacoes = {
        "original": audio,
        "lento": librosa.effects.time_stretch(
            audio,
            rate=0.92
        ),
        "rapido": librosa.effects.time_stretch(
            audio,
            rate=1.08
        ),
        "ruido": audio + np.random.normal(
            0,
            0.003,
            len(audio)
        )
    }

    for tipo, audio_variado in variacoes.items():

        audio_variado = np.clip(audio_variado, -1, 1)

        nome = f"{base}_{tipo}.wav"
        caminho = os.path.join(PASTA_FINAL, nome)

        sf.write(caminho, audio_variado, sr)

        registros_finais.append({
            "arquivo": nome,
            "caminho": caminho,
            "classe": linha["classe"],
            "texto": linha["texto"],
            "arquivo_original": linha["arquivo"],
            "tipo": tipo,
            "origem": "sintetico_azure_com_aumento_local"
        })

df_final = pd.DataFrame(registros_finais)

df_final.to_csv(
    os.path.join(PASTA_FINAL, "metadata.csv"),
    index=False,
    encoding="utf-8"
)

print(df_final["classe"].value_counts())
print("Total:", len(df_final))

classe
neutro                 400
tristeza               400
ansiedade_hesitacao    380
Name: count, dtype: int64
Total: 1180


In [13]:
import shutil
from google.colab import files

arquivo_zip = shutil.make_archive(
    "/content/dataset_audio_emocoes_1200",
    "zip",
    PASTA_FINAL
)

files.download(arquivo_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
%pip install -q openai-whisper librosa soundfile scikit-learn tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
import whisper
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

modelo_whisper = whisper.load_model("tiny")

df_meta = pd.read_csv(
    "/content/dataset_audio_emocoes_1200/metadata.csv"
)

resultados = []

for _, linha in tqdm(
    df_meta.iterrows(),
    total=len(df_meta)
):
    arquivo = linha["caminho"]

    audio, sr = librosa.load(
        arquivo,
        sr=16000,
        mono=True
    )

    transcricao = modelo_whisper.transcribe(
        arquivo,
        language="pt",
        fp16=False
    )

    texto = transcricao["text"].strip()
    quantidade_palavras = len(texto.split())

    duracao = librosa.get_duration(
        y=audio,
        sr=sr
    )

    rms = librosa.feature.rms(y=audio)[0]
    energia_media = float(np.mean(rms))

    intervalos = librosa.effects.split(
        audio,
        top_db=30
    )

    tempo_sonoro = sum(
        (fim - inicio) / sr
        for inicio, fim in intervalos
    )

    tempo_pausas = max(duracao - tempo_sonoro, 0)
    proporcao_pausas = tempo_pausas / duracao

    ritmo = (
        quantidade_palavras / (duracao / 60)
        if duracao > 0 else 0
    )

    f0, _, _ = librosa.pyin(
        audio,
        fmin=70,
        fmax=500,
        sr=sr
    )

    f0_valido = f0[~np.isnan(f0)]

    tom_medio = (
        float(np.mean(f0_valido))
        if len(f0_valido) > 0 else 0
    )

    tom_desvio = (
        float(np.std(f0_valido))
        if len(f0_valido) > 0 else 0
    )

    resultados.append({
        "arquivo": linha["arquivo"],
        "classe": linha["classe"],
        "grupo_original": linha["arquivo_original"],
        "transcricao_whisper": texto,
        "duracao": duracao,
        "energia_media": energia_media,
        "proporcao_pausas": proporcao_pausas,
        "ritmo_palavras_minuto": ritmo,
        "tom_medio_hz": tom_medio,
        "tom_desvio_hz": tom_desvio
    })

df_audio = pd.DataFrame(resultados)

df_audio.to_csv(
    "/content/caracteristicas_audio.csv",
    index=False,
    encoding="utf-8"
)

df_audio.head()

100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 172MiB/s]


  0%|          | 0/1180 [00:00<?, ?it/s]

,arquivo,classe,grupo_original,transcricao_whisper,duracao,energia_media,proporcao_pausas,ritmo_palavras_minuto,tom_medio_hz,tom_desvio_hz
0,ansiedade_hesitacao_000_original.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,um preciso de ajuda para ficar mais tranquilo.,4.183562,0.084545,0.242751,114.734751,272.751001,51.268402
1,ansiedade_hesitacao_000_lento.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,"Um, preciso de ajuda Mclooh!",4.547375,0.051451,0.218890,65.972127,268.475689,52.954579
2,ansiedade_hesitacao_000_rapido.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,Um. Esse здесь já ajuda a partir Mais um splas...,3.873687,0.061327,0.223479,170.380290,270.339237,54.597333
3,ansiedade_hesitacao_000_ruido.wav,ansiedade_hesitacao,ansiedade_hesitacao_000.wav,um preciso de ajuda para ficar mais tranquilo.,4.183562,0.085273,0.235102,114.734751,270.609201,53.022748
4,ansiedade_hesitacao_001_original.wav,ansiedade_hesitacao,ansiedade_hesitacao_001.wav,"Bem, preciso de ajuda para ficar mais tranquilo.",4.128188,0.077900,0.232593,116.273788,270.199429,57.908848


In [16]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

colunas = [
    "duracao",
    "energia_media",
    "proporcao_pausas",
    "ritmo_palavras_minuto",
    "tom_medio_hz",
    "tom_desvio_hz"
]

X = df_audio[colunas].fillna(0)
y = df_audio["classe"]
grupos = df_audio["grupo_original"]

divisor = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

indice_treino, indice_teste = next(
    divisor.split(X, y, groups=grupos)
)

modelo_audio = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

modelo_audio.fit(
    X.iloc[indice_treino],
    y.iloc[indice_treino]
)

previsoes = modelo_audio.predict(
    X.iloc[indice_teste]
)

print(classification_report(
    y.iloc[indice_teste],
    previsoes
))

                     precision    recall  f1-score   support

ansiedade_hesitacao       0.81      0.86      0.84        80
             neutro       0.86      0.79      0.82        80
           tristeza       0.97      1.00      0.99        76

           accuracy                           0.88       236
          macro avg       0.88      0.88      0.88       236
       weighted avg       0.88      0.88      0.88       236

